<a href="https://colab.research.google.com/github/mybright107/workflow_python/blob/main/getOCLCnumber_printbooks_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code for processing Worldcat API

# Python "requests", simple HTTP library. https://pypi.org/project/requests/
import requests

# Pandas data analysis library
import pandas as pd

# Allow access to files and folders in google drive
from google.colab import files
from google.colab import drive

# CSV library for importing and expoerting to CSV
import csv

# Datetime library for handling dates and times
import datetime

# Mount the google drive for access in the code
# Note: Google will ask for permission to access the google drive
drive.mount('/drive')


Mounted at /drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def getToken(refreshTkn=None):

    # Set up the base URL and credentials. These creds are Ron's
    base_token_url = 'https://oauth.oclc.org/token' # Base URL without query parameters
    client_id = '[YOUR CLINET ID]'
    client_secret = '[YOUR CLIENT SECRET]'

    # First check if this is already a refresh token. If not, we're generating a new one.
    if refreshTkn is None:
        # Initial token request using client_credentials grant type
        payload = {
            'scope': 'wcapi refresh_token', # space-separated scope for client_credentials
            'grant_type': 'client_credentials'
        }
        tokenResp = requests.post(base_token_url, auth=(client_id, client_secret), data=payload)
        tokenJson = tokenResp.json()
        apiToken = tokenJson["access_token"]
        refreshToken = tokenJson["refresh_token"]
        tokenExp = tokenJson["expires_at"]
        return apiToken, refreshToken, tokenExp;

    # Ok, this IS a refresh token.
    else:
        # Token refresh request using refresh_token grant type
        payload = {
            'refresh_token': refreshTkn,
            'grant_type': 'refresh_token'
        }
        tokenResp = requests.post(base_token_url, auth=(client_id, client_secret), data=payload)
        tokenJson = tokenResp.json()
        apiToken = tokenJson["access_token"]
        refreshToken = tokenJson["refresh_token"] # Should receive a new refresh token
        tokenExp = tokenJson["expires_at"]
        return apiToken, refreshToken, tokenExp;

In [ ]:
# Get OCLC Access Token and Refresh Token
OCLC_TOKEN, REFRESH_TOKEN, TOKEN_EXPIRY = getToken()

# Check if Google Drive is mounted, and if not, try to mount it again.
# This helps prevent 'Transport endpoint is not connected' errors when reading the input file.
import os
if not os.path.exists('/content/drive/MyDrive'): # Note: Check for '/content/drive/MyDrive' for the specific file path
    print("Google Drive not mounted, attempting to re-mount for input file...")
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive re-mounted for input file.")

data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/[YOUR CSV FILE]', encoding='latin1')


# Prepare results list
results = []

# Iterate over each row in the Excel
for _, row in data.iterrows():
    title = str(row["ti:"]).strip() if not pd.isna(row["ti:"]) else ""
    date_published = str(row["datePublished"]).strip() if not pd.isna(row["datePublished"]) else ""
    inLanguage = str(row["In:"]).strip() if not pd.isna(row["In:"]) else ""

    if not title:
        continue

    # Build query string
    query = f'ti:{title}'

    url = "https://americas.discovery.api.oclc.org/worldcat/search/v2/bibs"
    params = {
        "q": query,
        "inLanguage": inLanguage,
        "datePublished": date_published,   # ✅ separate param
        "inCatalogLanguage": "eng",
        "itemSubType": "book-printbook",
        "orderBy": "bestMatch",
        "limit": 1
    }

    max_retries = 3
    retries = 0
    while retries < max_retries:
        try:
            headers = {
                "Accept": "application/json",
                "Authorization": f"Bearer {OCLC_TOKEN}"
            }
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status() # This will raise HTTPError for 4xx/5xx responses
            data = response.json()

            oclc_num = ""
            main_titles = ""
            physical_description = ""

            if data.get("numberOfRecords", 0) > 0:
                record = data["bibRecords"][0]

                # ✅ FIXED: identifier is an object, not a list
                oclc_num = record.get("identifier", {}).get("oclcNumber", "")

                if "title" in record and "mainTitles" in record["title"]:
                    main_titles = "; ".join([t["text"] for t in record["title"]["mainTitles"]])

                physical_description = record.get("description", {}).get("physicalDescription", "")

            results.append({
                "Local_Title": title,
                "OCLC_Number": oclc_num,
                "OCLC_Title": main_titles,
                "Physical_Description": physical_description
            })

            print(f"✔ {title} → {oclc_num if oclc_num else 'No OCLC#'}")
            print("   URL:", response.url)
            break # Break out of the retry loop if successful

        except requests.exceptions.HTTPError as http_err:
            if http_err.response.status_code == 401:
                print(f"⚠️ 401 Unauthorized for {title}. Attempting to refresh token (retry {retries + 1}/{max_retries})...")
                OCLC_TOKEN, REFRESH_TOKEN, TOKEN_EXPIRY = getToken(REFRESH_TOKEN)
                retries += 1
                if retries == max_retries:
                    print(f"❌ Failed for {title} after {max_retries} retries: {http_err}")
                    results.append({
                        "Local_Title": title,
                        "OCLC_Number": "",
                        "OCLC_Title": "",
                        "Physical_Description": ""
                    })
                # Else, loop continues to retry
            else:
                print(f"❌ HTTP Error for {title}: {http_err}")
                results.append({
                    "Local_Title": title,
                    "OCLC_Number": "",
                    "OCLC_Title": "",
                    "Physical_Description": ""
                })
                break # Break out of retry loop for other HTTP errors
        except Exception as e:
            print(f"❌ General Error for {title}: {e}")
            results.append({
                "Local_Title": title,
                "OCLC_Number": "",
                "OCLC_Title": "",
                "Physical_Description": ""
            })
            break # Break out of retry loop for other general errors

# Save results to csv
out_df = pd.DataFrame(results)

# Check if Google Drive is mounted, and if not, try to mount it again.
# This helps prevent 'Transport endpoint is not connected' errors when writing the output file.
if not os.path.exists('/drive/My Drive'):
    print("Google Drive not mounted, attempting to re-mount for output file...")
    drive.mount('/drive', force_remount=True)
    print("Google Drive re-mounted for output file.")

# Output the dataframe to a CSV on our google drive. Output parameter of QUOTE_ALL. This means that the CSV
# will put double quotes around all of the data so it's all treated as text when importing. Sometimes long
# OCLC numbers (as well as other IDs, like MMSIDs) are mistaken for large numbers and converted to exponetial
# notation which tends to mess up IDs in Excel. Easier to treat it all as text and chenge to numbers as needed
# once it's in excel.
out_df.to_csv('/drive/My Drive/Colab Notebooks/[YOUR OUTPUT FILE NAME].csv', quoting=csv.QUOTE_ALL)